# 第09章：矩阵乘法（分块优化）— 数据复用的威力

## 前置知识
- 第01-06章（基础部分）
- 第06章中 `tl.dot` 的预览
- 矩阵乘法的基本数学定义：C[i,j] = Σ_k A[i,k] * B[k,j]

## 学习目标
- 理解朴素矩阵乘法的瓶颈（全局内存访问次数过多）
- 掌握**分块 (Tiling)** 策略的核心思想 —— 数据复用
- 实现一个标准的分块 GEMM kernel
- 分析 Block Size 对性能的影响
- 对比 CUDA 手写 kernel（`simt_regci` / `simt_smem`）与 Triton 的开发效率

## 对应 CUDA 代码
- `src/simt/01simt_regci.cu` — 寄存器增强计算强度
- `src/simt/02simt_smem.cu` — 使用 Shared Memory 做分块

In [1]:
import torch
import triton
import triton.language as tl
import time

## 9.1 回顾朴素版问题

在最朴素的矩阵乘法实现中，计算 C = A @ B (A: M×K, B: K×N, C: M×N)：

```
for i in range(M):       # 遍历 C 的每一行
    for j in range(N):   # 遍历 C 的每一列
        for k in range(K):   # 内积
            C[i,j] += A[i,k] * B[k,j]
```

### 问题：全局内存访问爆炸

```
计算 C[i,j] 需要:
  - 读取 A 的第 i 行:  K 次全局内存读取
  - 读取 B 的第 j 列:  K 次全局内存读取
  - 总共: 2K 次读取, K 次乘加

计算整个 C (M×N 个元素):
  - 总读取次数: M × N × 2K
  - 总计算次数: M × N × K (乘加)
  - 算术强度 = 计算量 / 访存量
              = (M×N×K) / (M×N×2K × sizeof(half))
              = 1 / (2 × 2) = 0.25 FLOP/Byte
```

**0.25 FLOP/Byte** 远低于 GPU 的计算/带宽比 (通常 > 50)，说明 kernel 严重受限于**内存带宽**。

### 关键观察：大量重复读取

```
A 的第 i 行被使用了多少次？
  → 计算 C[i,0], C[i,1], ..., C[i,N-1] 都需要 → 共 N 次！

B 的第 j 列被使用了多少次？
  → 计算 C[0,j], C[1,j], ..., C[M-1,j] 都需要 → 共 M 次！
```

如果我们能把 A 的行、B 的列缓存在快速存储中（寄存器 / Shared Memory），就能大幅减少全局内存访问。

## 9.2 分块策略 (Tiling)

### 核心思想

不再让每个线程独立计算单个 C[i,j]，而是让**一组线程（一个 program）**协作计算 C 的一个**子块**。

### 分块示意图

```
矩阵 A (M × K)                  矩阵 B (K × N)               矩阵 C (M × N)
┌─────────────────────┐          ┌─────────────────────┐      ┌─────────────────────┐
│                     │          │     BLOCK_N         │      │     BLOCK_N         │
│    BLOCK_K          │          │    ◄──────►         │      │    ◄──────►         │
│   ◄──────►          │          │ ┌─────────┐         │      │ ┌─────────┐         │
│ ┌─────────┐         │  BLOCK_K │ │         │         │      │ │         │         │
│ │  A_tile  │ BLOCK_M│  ▲       │ │ B_tile  │         │      │ │  C_blk  │ BLOCK_M │
│ │ (m × k)  │ ▲      │  │       │ │ (k × n) │         │      │ │ (m × n) │ ▲       │
│ │         │ │      │  ▼       │ │         │         │      │ │         │ │       │
│ └─────────┘ ▼      │          │ └─────────┘         │      │ └─────────┘ ▼       │
│                     │          │                     │      │                     │
└─────────────────────┘          └─────────────────────┘      └─────────────────────┘

一个 program 的工作:
  C_blk[m,n] = Σ_{k_step} A_tile[m,k] @ B_tile[k,n]
```

### K 维度上的迭代

```
K 维度被划分为 K/BLOCK_K 个步骤：

步骤 0:                步骤 1:                步骤 2:           ...
                                                                        
A[:, 0:BK]             A[:, BK:2BK]           A[:, 2BK:3BK]    ...      
┌──────┐               ┌──────┐               ┌──────┐                  
│A_tile│               │A_tile│               │A_tile│                  
│  0   │               │  1   │               │  2   │                  
└──────┘               └──────┘               └──────┘                  
    @                      @                      @                      
┌──────┐               ┌──────┐               ┌──────┐                  
│B_tile│               │B_tile│               │B_tile│                  
│  0   │               │  1   │               │  2   │                  
└──────┘               └──────┘               └──────┘                  
B[0:BK, :]             B[BK:2BK, :]           B[2BK:3BK, :]    ...      
                                                                        
    ↓                      ↓                      ↓                      
    ───────────── acc (累加器, FP32) ─────────────                       
                        C_blk                                            
```

### 数据复用的关键

```
同一个 A_tile (BLOCK_M × BLOCK_K) 被哪些 program 使用？
  → 同一行的所有 program：pid_m 相同, pid_n = 0, 1, ..., N/BLOCK_N - 1
  → 复用次数 = N / BLOCK_N

同一个 B_tile (BLOCK_K × BLOCK_N) 被哪些 program 使用？
  → 同一列的所有 program：pid_n 相同, pid_m = 0, 1, ..., M/BLOCK_M - 1
  → 复用次数 = M / BLOCK_M
```

虽然不同 program 之间不直接共享数据（不像 CUDA 的 shared memory），但 Triton 编译器会利用
L2 cache 来实现跨 program 的数据复用。**在 program 内部**，同一个 tile 被用于计算 BLOCK_M × BLOCK_N
个输出，这才是分块带来的核心收益。

## 9.3 数据复用分析

### 不使用分块（朴素版）

```
每个输出元素 C[i,j]:
  读取: A 的第 i 行 (K 个元素) + B 的第 j 列 (K 个元素) = 2K
  计算: K 次乘加 = 2K FLOPs

整个 C (M×N 个元素):
  总读取: M × N × 2K 个 FP16 元素 = M × N × 2K × 2 Bytes
  总计算: M × N × 2K FLOPs
  算术强度: (M×N×2K) / (M×N×2K×2) = 0.5 FLOP/Byte
```

### 使用分块

```
每个 program 计算一个 (BLOCK_M × BLOCK_N) 的 C 子块:
  K 维度分为 K/BLOCK_K 步
  每步读取:
    A_tile: BLOCK_M × BLOCK_K 个元素
    B_tile: BLOCK_K × BLOCK_N 个元素
  每步计算:
    BLOCK_M × BLOCK_N × BLOCK_K × 2 FLOPs (乘和加)

一个 program 的总量:
  总读取: (K/BLOCK_K) × (BLOCK_M × BLOCK_K + BLOCK_K × BLOCK_N) × 2 Bytes
        = K × (BLOCK_M + BLOCK_N) × 2 Bytes
  总计算: BLOCK_M × BLOCK_N × K × 2 FLOPs

算术强度:
  = (BLOCK_M × BLOCK_N × K × 2) / (K × (BLOCK_M + BLOCK_N) × 2)
  = BLOCK_M × BLOCK_N / (BLOCK_M + BLOCK_N)
```

### 具体数值

| BLOCK_M | BLOCK_N | 算术强度 (FLOP/Byte) | 相比朴素版提升 |
|---------|---------|---------------------|---------------|
| 16      | 16      | 8                   | 16x           |
| 32      | 32      | 16                  | 32x           |
| 64      | 64      | 32                  | 64x           |
| 128     | 128     | 64                  | 128x          |
| 128     | 256     | 85.3                | 171x          |

**分块越大，数据复用越多，算术强度越高。** 但受限于寄存器和 Shared Memory 大小。

对比 CUDA 项目中的参数：
- `simt_regci`: BlockTileM=256, BlockTileN=128, 没有 K 分块 → 每个 k 步都从全局内存读
- `simt_smem`: BlockTileM=256, BlockTileN=128, BlockTileK=16 → 通过 shared memory 实现 tile 内复用

## 9.4 实现 — 标准分块 GEMM Kernel

下面实现一个标准的分块矩阵乘法。对比 CUDA 中需要手动管理 shared memory、线程索引、数据搬运等细节，
Triton 只需要关注 **program 级别的逻辑**。

### 参数设计
```
输入: A (M×K, FP16), B (K×N, FP16)
输出: C (M×N, FP16)
累加器: FP32 (避免 FP16 精度损失)

grid: (M/BLOCK_M, N/BLOCK_N)  — 2D 网格
每个 program 计算 C 的一个 (BLOCK_M × BLOCK_N) 子块
```

In [2]:
@triton.jit
def matmul_blocked_kernel(
    # 矩阵指针
    a_ptr, b_ptr, c_ptr,
    # 矩阵维度
    M, N, K,
    # A 的行步长 (= K), B 的行步长 (= N), C 的行步长 (= N)
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    # 分块大小 (编译时常量)
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    分块矩阵乘法: C = A @ B
    
    对应 CUDA 代码:
    - simt_regci: 每个 thread 计算 ThreadTileM x ThreadTileN 的 C 子块，无 smem
    - simt_smem:  block 协作将 A/B tile 搬到 shared memory，然后计算
    
    Triton 版本: 编译器自动处理 smem / register tiling / 数据搬运
    """
    # ========== 1. 确定当前 program 负责的 C 子块位置 ==========
    pid_m = tl.program_id(0)  # C 子块的行索引
    pid_n = tl.program_id(1)  # C 子块的列索引
    
    # ========== 2. 计算当前 program 对应的行/列偏移 ==========
    # rm: 当前子块在 M 维度的行索引数组 [pid_m*BM, pid_m*BM+1, ..., pid_m*BM+BM-1]
    # rn: 当前子块在 N 维度的列索引数组 [pid_n*BN, pid_n*BN+1, ..., pid_n*BN+BN-1]
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # (BLOCK_M,)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # (BLOCK_N,)
    rk = tl.arange(0, BLOCK_K)                     # (BLOCK_K,)
    
    # ========== 3. 计算 A_tile 和 B_tile 的初始指针 ==========
    # A_tile: A[rm, 0:BLOCK_K] → 指针 = a_ptr + rm[:,None]*stride_am + rk[None,:]*stride_ak
    # B_tile: B[0:BLOCK_K, rn] → 指针 = b_ptr + rk[:,None]*stride_bk + rn[None,:]*stride_bn
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak  # (BLOCK_M, BLOCK_K)
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn  # (BLOCK_K, BLOCK_N)
    
    # ========== 4. 初始化 FP32 累加器 ==========
    # 对应 CUDA: float4 reg_c[...]{0};
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)  # (BLOCK_M, BLOCK_N)
    
    # ========== 5. K 维度循环 —— 逐 tile 累加 ==========
    # 对应 CUDA simt_smem: for (int k = 0; k < K; k += BlockTileK)
    for k_start in range(0, K, BLOCK_K):
        # 加载 A_tile (BLOCK_M × BLOCK_K), 类型为 FP16
        # mask 处理边界 (当 M 或 K 不是 BLOCK 的整数倍时)
        a_tile = tl.load(
            a_ptrs,
            mask=(rm[:, None] < M) & (rk[None, :] + k_start < K),
            other=0.0,
        )  # (BLOCK_M, BLOCK_K)
        
        # 加载 B_tile (BLOCK_K × BLOCK_N), 类型为 FP16
        b_tile = tl.load(
            b_ptrs,
            mask=(rk[:, None] + k_start < K) & (rn[None, :] < N),
            other=0.0,
        )  # (BLOCK_K, BLOCK_N)
        
        # 矩阵乘并累加: acc += A_tile @ B_tile
        # 对应 CUDA simt_smem 中的:
        #   for (int bk = 0; bk < BlockTileK; bk++)
        #     reg_c[i*ldm_regC + j] += reg_a[i] * reg_b[j]
        # Triton 只需要一行！编译器自动映射到 Tensor Core
        acc = tl.dot(a_tile, b_tile, acc=acc)
        
        # 移动到下一个 K tile
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    
    # ========== 6. 将结果从 FP32 转回 FP16 并写入全局内存 ==========
    c = acc.to(tl.float16)
    
    # C 的指针
    c_ptrs = c_ptr + rm[:, None] * stride_cm + rn[None, :] * stride_cn
    c_mask = (rm[:, None] < M) & (rn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

In [3]:
def matmul_blocked(a: torch.Tensor, b: torch.Tensor,
                   BLOCK_M=128, BLOCK_N=128, BLOCK_K=32) -> torch.Tensor:
    """
    分块矩阵乘法的 host 端包装函数。
    
    Args:
        a: (M, K) FP16 tensor
        b: (K, N) FP16 tensor
    Returns:
        c: (M, N) FP16 tensor
    """
    assert a.dtype == torch.float16 and b.dtype == torch.float16
    assert a.shape[1] == b.shape[0], f"K 维度不匹配: {a.shape[1]} vs {b.shape[0]}"
    
    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    
    # 2D grid: (M方向的block数, N方向的block数)
    # 对应 CUDA: dim3 grid(divCeil(N, BlockTileN), divCeil(M, BlockTileM))
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    
    matmul_blocked_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_K=BLOCK_K,
    )
    return c

In [4]:
# ========== 正确性验证 ==========
torch.manual_seed(42)

M, N, K = 2048, 2048, 1024  # 与 CUDA 项目中的测试参数一致
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

# Triton 分块实现
c_triton = matmul_blocked(a, b)

# PyTorch 参考实现 (cuBLAS)
c_ref = torch.matmul(a, b)

# FP16 精度较低，使用较大的 tolerance
print(f"矩阵规模: A({M}×{K}) @ B({K}×{N}) = C({M}×{N})")
print(f"最大绝对误差: {(c_triton - c_ref).abs().max().item():.4f}")
print(f"平均绝对误差: {(c_triton - c_ref).abs().mean().item():.4f}")
print(f"相对误差 (L2): {torch.norm(c_triton - c_ref) / torch.norm(c_ref):.6f}")
print(f"正确性检查 (atol=1.0): {torch.allclose(c_triton, c_ref, atol=1.0)}")

矩阵规模: A(2048×1024) @ B(1024×2048) = C(2048×2048)
最大绝对误差: 0.0000
平均绝对误差: 0.0000
相对误差 (L2): 0.000000
正确性检查 (atol=1.0): True


### Kernel 逐行解析

让我们回顾 kernel 的关键步骤，并对比 CUDA 实现：

| 步骤 | Triton | CUDA (simt_smem) |
|------|--------|------------------|
| 确定子块位置 | `pid_m = tl.program_id(0)` | `blockIdx.x`, `blockIdx.y` |
| 计算偏移 | `rm = pid_m * BM + tl.arange(0, BM)` | `block_offsety = blockIdx.y * BlockTileM` |
| 加载 A tile | `tl.load(a_ptrs, mask=...)` | 手动从 global → smem，再从 smem → reg |
| 加载 B tile | `tl.load(b_ptrs, mask=...)` | 同上，需要 `__syncthreads()` |
| 矩阵乘累加 | `acc = tl.dot(a, b, acc=acc)` | 双重循环 `reg_c[i] += reg_a[i] * reg_b[j]` |
| 写回结果 | `tl.store(c_ptrs, c, mask=...)` | 手动 float4 写回 |

**Triton 的优势**：编译器自动处理了 shared memory 的分配、数据搬运、bank conflict 避免、
线程同步 (`__syncthreads`)、Tensor Core 映射等底层细节。

## 9.5 Block Size 对性能的影响

不同的 BLOCK_M / BLOCK_N / BLOCK_K 组合会影响：

1. **数据复用率**：Block 越大，每个元素被复用次数越多（算术强度越高）
2. **并行度**：Block 越大，grid 中的 program 越少，GPU SM 可能利用不满
3. **寄存器/Shared Memory 压力**：Block 越大，需要更多 on-chip 资源
4. **向量化效率**：BLOCK_K 影响内存访问的向量化

让我们实测不同配置的性能。

In [5]:
def benchmark_matmul(M, N, K, BLOCK_M, BLOCK_N, BLOCK_K, num_warmup=10, num_rep=50):
    """
    测量分块 GEMM 的性能。
    返回: (时间 ms, TFLOPS)
    """
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    c = torch.empty(M, N, device='cuda', dtype=torch.float16)
    
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    
    # Warmup
    for _ in range(num_warmup):
        matmul_blocked_kernel[grid](
            a, b, c, M, N, K,
            a.stride(0), a.stride(1),
            b.stride(0), b.stride(1),
            c.stride(0), c.stride(1),
            BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        )
    torch.cuda.synchronize()
    
    # Benchmark
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    for _ in range(num_rep):
        matmul_blocked_kernel[grid](
            a, b, c, M, N, K,
            a.stride(0), a.stride(1),
            b.stride(0), b.stride(1),
            c.stride(0), c.stride(1),
            BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        )
    end.record()
    torch.cuda.synchronize()
    
    elapsed_ms = start.elapsed_time(end) / num_rep
    flops = 2.0 * M * N * K  # 乘加 = 2 FLOPs
    tflops = flops / (elapsed_ms * 1e-3) / 1e12
    
    return elapsed_ms, tflops

In [6]:
# ========== 不同 Block Size 的性能对比 ==========
M, N, K = 2048, 2048, 1024

configs = [
    # (BLOCK_M, BLOCK_N, BLOCK_K)
    (32,  32,  16),
    (32,  32,  32),
    (64,  64,  16),
    (64,  64,  32),
    (128, 128, 16),
    (128, 128, 32),
    (128, 128, 64),
    (128, 256, 32),
    (256, 128, 32),
]

print(f"矩阵规模: M={M}, N={N}, K={K}")
print(f"{'BLOCK_M':>8} {'BLOCK_N':>8} {'BLOCK_K':>8} | {'Grid':>12} | {'算术强度':>10} | {'Time(ms)':>10} {'TFLOPS':>8}")
print("-" * 85)

results = []
for bm, bn, bk in configs:
    try:
        grid_m, grid_n = triton.cdiv(M, bm), triton.cdiv(N, bn)
        arith_intensity = bm * bn / (bm + bn)  # FLOP/Byte 的简化指标
        elapsed_ms, tflops = benchmark_matmul(M, N, K, bm, bn, bk)
        results.append((bm, bn, bk, elapsed_ms, tflops))
        print(f"{bm:>8} {bn:>8} {bk:>8} | {f'{grid_m}x{grid_n}':>12} | {arith_intensity:>10.1f} | {elapsed_ms:>10.3f} {tflops:>8.2f}")
    except Exception as e:
        print(f"{bm:>8} {bn:>8} {bk:>8} | {'ERROR':>12} | {str(e)[:40]}")

if results:
    best = min(results, key=lambda x: x[3])
    print(f"\n最优配置: BLOCK_M={best[0]}, BLOCK_N={best[1]}, BLOCK_K={best[2]}")
    print(f"  时间: {best[3]:.3f} ms, 性能: {best[4]:.2f} TFLOPS")

矩阵规模: M=2048, N=2048, K=1024
 BLOCK_M  BLOCK_N  BLOCK_K |         Grid |       算术强度 |   Time(ms)   TFLOPS
-------------------------------------------------------------------------------------
      32       32       16 |        64x64 |       16.0 |      0.093    91.89
      32       32       32 |        64x64 |       16.0 |      0.074   115.37
      64       64       16 |        32x32 |       32.0 |      0.044   197.27
      64       64       32 |        32x32 |       32.0 |      0.055   154.87
     128      128       16 |        16x16 |       64.0 |      0.031   274.05
     128      128       32 |        16x16 |       64.0 |      0.033   259.27
     128      128       64 |        16x16 |       64.0 |      0.040   217.37
     128      256       32 |         16x8 |       85.3 |      0.128    66.89
     256      128       32 |         8x16 |       85.3 |      0.160    53.68

最优配置: BLOCK_M=128, BLOCK_N=128, BLOCK_K=16
  时间: 0.031 ms, 性能: 274.05 TFLOPS


### 性能分析

从 benchmark 结果中我们可以观察到：

1. **BLOCK 太小 (32×32)**：算术强度低 (16)，大量时间花在内存访问上
2. **BLOCK 太大 (256×128)**：虽然算术强度高，但可能因为寄存器溢出 (register spilling) 而变慢
3. **中等 BLOCK (64×64 或 128×128)**：通常是最佳平衡点
4. **BLOCK_K 的影响**：较大的 BLOCK_K 可以减少 K 循环次数，但增加单步的内存需求

```
性能随 Block Size 变化的趋势:

TFLOPS
  ▲
  │        ╭──╮
  │       ╱    ╲
  │     ╱        ╲
  │   ╱            ╲
  │ ╱                ╲
  │╱                   ╲
  ┼───────────────────────►
  小                    大
       Block Size

  ◄─── 带宽受限 ───►◄── 资源受限 ──►
  算术强度不足         寄存器溢出
  并行度高但复用少     复用多但SM利用不满
```

## 9.6 CUDA 对比

### 性能数据 (M=N=2048, K=1024, RTX PRO 6000 Blackwell, SM 12.0, 188 SMs)

| 实现 | 时间 (ms) | TFLOPS | 核心策略 | 代码量 |
|------|-----------|--------|---------|--------|
| **simt_regci** (CUDA) | 0.3833 | 22.41 | 寄存器增强计算强度, 无 smem, 直接从 global → reg | ~100 行 |
| **simt_smem** (CUDA) | 0.2518 | 34.11 | Shared Memory 分块, global → smem → reg | ~160 行 |
| **Triton 分块版** | ~0.04-0.10* | — | 编译器自动 tiling + Tensor Core | ~40 行 |
| **cuBLAS** | 0.1384 | 62.05 | 高度优化的库实现 | 1 行调用 |

\* Triton 性能因 Block Size 配置而异，此处为 kernel-only 时间 (数据已在显存)

### 开发效率对比

| 方面 | CUDA (simt_smem) | Triton |
|------|-----------------|--------|
| Shared Memory 管理 | 手动声明 `__shared__`, 手动搬运 | 编译器自动 |
| 线程索引计算 | `threadIdx.x`, `threadIdx.y`, 手动映射 | 不需要，program 级别抽象 |
| Bank Conflict | 需要手动 padding 或 swizzle | 编译器自动处理 |
| 同步 | 手动 `__syncthreads()` | 编译器自动插入 |
| 向量化 (float4) | 手动 `reinterpret_cast<float4*>` | 编译器自动 |
| Tensor Core | 需要 `wmma` 或内联 PTX | `tl.dot` 自动映射 |
| 边界处理 | 手动 if 判断或 padding | `mask` 参数 |

### CUDA 代码片段对比

```cpp
// CUDA simt_smem: 加载 A tile 到 shared memory (~10 行)
for (int i = tid; i < BlockTileM * ldm_blockA_f4size; i += blockDim.x * blockDim.y) {
    int offset_ld2s_global_ax = i % ldm_blockA_f4size;
    int offset_ld2s_global_ay = i / ldm_blockA_f4size;
    float4 buffer = gmem_blockA_f4_ptr[k/8 + offset_ld2s_global_ay * ldm_A_f4size + ...];
    smem_blockA_f4_ptr[offset_st_smem_ay * ldm_blockA_f4size + offset_st_smem_ax] = buffer;
}
__syncthreads();
```

```python
# Triton: 加载 A tile (1 行！)
a_tile = tl.load(a_ptrs, mask=(rm[:, None] < M) & (rk[None, :] + k_start < K), other=0.0)
```

**结论**：Triton 以接近 CUDA 手写 kernel 的性能，提供了数倍的开发效率提升。

In [7]:
# ========== Triton vs cuBLAS (torch.matmul) 对比 ==========
M, N, K = 2048, 2048, 1024
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

# cuBLAS baseline
for _ in range(10):
    _ = torch.matmul(a, b)
torch.cuda.synchronize()

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
num_rep = 50

start.record()
for _ in range(num_rep):
    c_cublas = torch.matmul(a, b)
end.record()
torch.cuda.synchronize()
cublas_ms = start.elapsed_time(end) / num_rep

# Triton best config
triton_ms, triton_tflops = benchmark_matmul(M, N, K, 128, 128, 32)

flops = 2.0 * M * N * K
cublas_tflops = flops / (cublas_ms * 1e-3) / 1e12

print(f"矩阵规模: M={M}, N={N}, K={K}")
print(f"{'实现':>20} | {'时间 (ms)':>10} | {'TFLOPS':>8} | {'相对 cuBLAS':>12}")
print("-" * 65)
print(f"{'cuBLAS (torch.matmul)':>20} | {cublas_ms:>10.3f} | {cublas_tflops:>8.2f} | {'1.00x':>12}")
print(f"{'Triton 分块 (128x128)':>20} | {triton_ms:>10.3f} | {triton_tflops:>8.2f} | {cublas_ms/triton_ms:>11.2f}x")
print(f"{'CUDA simt_regci':>20} | {'0.3833':>10} | {'22.41':>8} | {'Blackwell 实测':>12}")
print(f"{'CUDA simt_smem':>20} | {'0.2518':>10} | {'34.11':>8} | {'Blackwell 实测':>12}")

矩阵规模: M=2048, N=2048, K=1024
                  实现 |    时间 (ms) |   TFLOPS |    相对 cuBLAS
-----------------------------------------------------------------
cuBLAS (torch.matmul) |      0.033 |   258.48 |        1.00x
 Triton 分块 (128x128) |      0.033 |   259.47 |        1.00x
     CUDA simt_regci |     0.3833 |    22.41 | Blackwell 实测
      CUDA simt_smem |     0.2518 |    34.11 | Blackwell 实测


## 9.7 总结

### 本章要点

1. **朴素矩阵乘法的瓶颈**：每个输出元素独立计算，导致大量重复的全局内存读取，算术强度极低 (~0.5 FLOP/Byte)

2. **分块策略的核心思想**：将 C 划分为子块，每个 program 负责一个子块，沿 K 维度逐 tile 加载 A 和 B 并累加

3. **数据复用**：
   - program 内部：一个 A_tile 的每一行被用于计算 BLOCK_N 个输出 → 复用 BLOCK_N 次
   - 算术强度从 0.5 提升到 BLOCK_M × BLOCK_N / (BLOCK_M + BLOCK_N)

4. **Triton 实现的简洁性**：
   - 核心逻辑只有 ~40 行 Python
   - 编译器自动处理 shared memory、线程映射、Tensor Core 调用
   - 对比 CUDA 手写版 (simt_smem) 减少了 3-4 倍代码

5. **Block Size 的权衡**：
   - 太小 → 算术强度低，带宽受限
   - 太大 → 寄存器溢出，并行度不足
   - 最佳值通常在 64-128 范围

### 与 CUDA 项目的对应关系

```
Triton 分块 GEMM
    ├── pid_m, pid_n        ←→  blockIdx.x, blockIdx.y
    ├── tl.load A_tile      ←→  global → smem 搬运 + __syncthreads()
    ├── tl.load B_tile      ←→  global → smem 搬运 + __syncthreads()
    ├── tl.dot(a, b, acc)   ←→  reg_c += reg_a * reg_b (双重循环)
    └── tl.store            ←→  float4 写回 global memory
```

### 练习

1. **边界处理验证**：测试 M=2000, N=2000, K=1000 (非 Block 整数倍) 是否正确
2. **BF16 版本**：将输入改为 `torch.bfloat16`，观察精度和性能变化
3. **非对称 Block**：尝试 BLOCK_M != BLOCK_N (如 128×64)，在不同形状的矩阵上测试
4. **转置 B**：修改 kernel 支持 B 为列主序 (B^T)，观察对性能的影响
5. **思考题**：为什么 Triton 不需要 `__syncthreads()`？（提示：program 的抽象层次）

### 下一章预告

第10章将引入 **L2 Cache 优化 (Swizzle)**，通过调整 program 的发射顺序来提高 L2 cache 命中率，
这对应 CUDA 中更高级的 grid 调度技巧。

In [8]:
# ========== 练习 1: 边界处理验证 ==========
print("练习 1: 非对齐矩阵尺寸")
for M, N, K in [(2000, 2000, 1000), (1023, 513, 777), (100, 200, 300)]:
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    c_triton = matmul_blocked(a, b, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32)
    c_ref = torch.matmul(a, b)
    
    max_err = (c_triton - c_ref).abs().max().item()
    passed = max_err < 2.0  # FP16 误差容忍
    print(f"  ({M:>4}×{K:>4}) @ ({K:>4}×{N:>4}): max_err={max_err:.4f} {'PASS' if passed else 'FAIL'}")

练习 1: 非对齐矩阵尺寸
  (2000×1000) @ (1000×2000): max_err=0.0625 PASS
  (1023× 777) @ ( 777× 513): max_err=0.0625 PASS
  ( 100× 300) @ ( 300× 200): max_err=0.0625 PASS
